In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Clone repo in Colab if needed
if IS_COLAB and not os.path.exists("computer-data-analysis-report"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/Aidas-dev/computer-data-analysis-report.git

# Change to repo directory
repo_dir = "computer-data-analysis-report"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull -q
else:
    !cd {repo_dir} 2>/dev/null || echo "Repo not found"

# Install uv (faster than pip)
!pip install uv -q

# Install dependencies with uv (bypasses Colab's pip resolver issues)
!uv pip install --system -r requirements.txt -q

# Setup DVC in Colab
if IS_COLAB:
    from google.colab import userdata
    try:
        access_key = userdata.get("OCI_ACCESS_KEY")
        secret_key = userdata.get("OCI_SECRET_KEY")
        !dvc remote modify --local oracle_remote access_key_id {access_key}
        !dvc remote modify --local oracle_remote secret_access_key {secret_key}
        !dvc pull -q
    except:
        pass  # Secrets not configured, skip DVC pull

print("✅ Environment ready!")


# Yahoo Finance Data Extraction & DVC Tracking

This notebook demonstrates how to fetch data using `yfinance` and then use our custom `src/data/dvc_storage.py` module to store the resulting DataFrames securely in the Oracle Cloud DVC remote.

**Prerequisite:** Make sure you have successfully authenticated with Oracle via DVC (run `00-colab-setup.ipynb` if you are in Colab).

In [ ]:
import yfinance as yf
import pandas as pd
import sys
import os

# Ensure the 'src' directory is accessible for imports
sys.path.append(os.path.abspath('..'))

from src.data.dvc_storage import store_dataframes_in_dvc

In [ ]:
# Example 1: Fetch and store a single ticker
ticker_name = 'MSFT'
print(f"📉 Fetching 1y data for {ticker_name}...")
msft_df = yf.Ticker(ticker_name).history(period='1y')

if not msft_df.empty:
    # We pass the single DataFrame and its name to our storage function
    store_dataframes_in_dvc(data=msft_df, names=ticker_name, folder_type='raw')
else:
    print(f"❌ No data returned for {ticker_name}")

In [ ]:
# Example 2: Fetch and store multiple tickers simultaneously as a list
tickers_list = ['GOOGL', 'AMZN']
dfs = []

print(f"\n📉 Fetching 1mo data for multiple tickers: {tickers_list}...")
for t in tickers_list:
    df = yf.Ticker(t).history(period='1mo')
    dfs.append(df)

# Ensure we have valid data before storing
valid_dfs = [df for df in dfs if not df.empty]
valid_names = [t for t, df in zip(tickers_list, dfs) if not df.empty]

if valid_dfs:
    # We pass the list of DataFrames and the list of names to our storage function
    store_dataframes_in_dvc(data=valid_dfs, names=valid_names, folder_type='raw')
else:
    print("❌ No valid data returned for the tickers list.")